In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Schema drift check: customers
# MAGIC
# MAGIC A plain notebook, NOT a DLT pipeline file - this is a periodic
# MAGIC diagnostic check, not transformation logic. Run manually, or attach
# MAGIC to its own lightweight scheduled Job (e.g. daily) to get an alert if
# MAGIC Postgres's `customers` table shape ever changes underneath you.
# MAGIC
# MAGIC Reads from `landing_customers` (the raw, unparsed Kafka capture) so
# MAGIC this check sees the TRUE current shape of the data, independent of
# MAGIC whatever bronze_kafka.py's hardcoded StructType currently expects.

# COMMAND ----------

import json
from pyspark.sql import functions as F

# Must match bronze_kafka.py's customer_schema field names exactly - update
# BOTH places if the pipeline's expected schema is ever deliberately changed.
EXPECTED_CUSTOMER_FIELDS = {
    "customer_id", "name", "email", "signup_date", "phone",
    "date_of_birth", "loyalty_tier", "updated_at",
}

LANDING_TABLE = "main.bronze.landing_customers"  # adjust catalog/schema if different

# COMMAND ----------


def check_customer_schema_drift(landing_df, expected_fields, sample_size=200):
    sample_rows = (
        landing_df
        .select(F.get_json_object(F.col("value"), "$.payload.after").alias("after_json"))
        .filter(F.col("after_json").isNotNull())
        .orderBy(F.col("timestamp").desc())  # most recent messages first
        .limit(sample_size)
        .collect()
    )

    if not sample_rows:
        print("No recent messages found to check - is the pipeline running?")
        return None

    actual_fields = set()
    for row in sample_rows:
        actual_fields |= set(json.loads(row["after_json"]).keys())

    new_fields = actual_fields - expected_fields
    missing_fields = expected_fields - actual_fields

    if new_fields or missing_fields:
        print(f"SCHEMA DRIFT DETECTED on customers (checked {len(sample_rows)} recent messages):")
        if new_fields:
            print(f"  NEW fields (in live Postgres data, not yet in the pipeline schema): {new_fields}")
        if missing_fields:
            print(f"  MISSING fields (pipeline expects these, live data no longer has them): {missing_fields}")
        print("\n  Action needed: update customer_schema in bronze_kafka.py, or investigate "
              "whether this is an unexpected upstream change in Postgres.")
        return False
    else:
        print(f"No schema drift detected - checked {len(sample_rows)} recent messages, all match.")
        return True

# COMMAND ----------

landing_df = spark.table(LANDING_TABLE)
check_customer_schema_drift(landing_df, EXPECTED_CUSTOMER_FIELDS)